# Annotator evaluation for the theory-pluralist suspense study

This notebook answers two questions about the annotations, in order:

**Q1 — Reliability.** When annotators use the *same* operationalization, how consistently do
they rate sentences? This is standard inter-annotator agreement (IAA), reported per theory and
overall, with the metric best suited to ordinal, zero-inflated ratings.

**Q2 — Does theory matter? (your supervisor's question).** Do annotators who share a theory
rank sentences for suspense/anxiety *more similarly* than annotators who used different theories?
If yes, the four operationalizations are producing distinguishable annotation behaviour — the
premise of the theory-pluralist design. This is the intra- vs. inter-theory coherence test from
the email: Spearman rank correlation per annotator pair per text, averaged, `intra − inter`, with
a permutation test for significance.

The two questions use different tools on purpose. **Q1 wants chance-corrected agreement**
(Krippendorff's alpha etc.); **Q2 wants a scale-invariant ordering measure** (Spearman) and, as
the email notes, *no* chance correction, because the comparison is intra vs. inter, not against a
gold standard.

---

**How the data is read.** One folder per text, each holding the three `Group_*.xlsx` files.
Every worksheet is named `T<theory><group>_<Text>` (e.g. `T1a_...` = theory 1, group *a*), so a
`(group, theory)` sheet is one annotation **unit**. Two units are an **intra**-theory pair when
their theory matches, **inter** otherwise. Group O's finer sentence split is reconciled by
aligning on `Sentence_ID`.

**Design specifics already found in your data**, surfaced again where they matter below:
`T4c` is an extra completed theory-4 sheet in Group O; some `character_anxiety_level` units are
constant (all-zero), which makes chance-corrected agreement undefined for them; and each group
annotated under all four theories, so some inter-theory pairs share a group.

## 0 · Setup and configuration

`ROOT` is the parent folder of the per-text sub-folders. Everything else has a sensible default;
the analysis choices that could affect your supervisor's numbers are grouped and commented.

In [41]:
# ── OUTPUT LOGGING SETUP ─────────────────────────────────────────────
import sys, io
from pathlib import Path
from datetime import datetime

_output_log = io.StringIO()

class _Tee:
    def __init__(self, *streams): self.streams = streams
    def write(self, data):
        for s in self.streams: s.write(data)
    def flush(self):
        for s in self.streams: s.flush()

_original_stdout = sys.stdout
sys.stdout = _Tee(_original_stdout, _output_log)

In [42]:
# --- edit this ---------------------------------------------------------------
ROOT = Path("/Users/sguhr/Desktop/20260720_IAA_Evaluation")   # parent of per-text sub-folders

# --- defaults ---------------------------------------------------------------
#DEFAULT_TARGET = "reader_suspense_level"     # or "character_anxiety_level"
DEFAULT_TARGET = "character_anxiety_level"
# --- Q2 (coherence) analysis choices ----------------------------------------
STATISTIC = "intra_minus_inter"              # positive => intra more coherent (email's sanity check)
N_PERMUTATIONS = 10_000
RANDOM_SEED = 20260720
EXCLUDE_SAME_GROUP_INTER = False             # True => drop inter pairs that share a group

# --- data handling ----------------------------------------------------------
EXCLUDE_GROUPS = set()                        # e.g. {"c"} to drop the T4c sheet everywhere
COMPLETENESS_MIN_FRACTION = 0.50              # a unit "completed" a text at/above this share
MIN_SHARED_SENTENCES = 3                      # minimum overlap to compute a pair
SAVE_DIR = Path("/Users/sguhr/Desktop/20260720_IAA_Evaluation/evaluation_outputs")         # export target

In [43]:
import re, glob, warnings
from itertools import combinations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

warnings.filterwarnings("ignore")
sns.set_theme(style="white")
SHEET_RE = re.compile(r"^T(\d+)([A-Za-z])_(.+)$")
COL_ALIASES = {"reader_suspense_level": ["reader_suspense_level"],
               "character_anxiety_level": ["character_anxiety_level"]}
CHANCE_CORRECTED = {"Krippendorff (ordinal)", "Cohen kappa", "Cohen kappa (weighted)"}

### Agreement metrics (self-contained, validated)

Inlined so the notebook needs no external agreement package; each was checked against the
`krippendorff` and `scikit-learn` reference implementations to machine precision.

In [44]:
def krippendorff_alpha(data, level="ordinal"):
    """data: raters x items, NaN = missing."""
    data = np.asarray(data, dtype=float)
    values = np.unique(data[~np.isnan(data)])
    if values.size < 2:
        return np.nan
    v_index = {v: i for i, v in enumerate(values)}; V = len(values)
    o = np.zeros((V, V))
    for col in data.T:
        r = col[~np.isnan(col)]; m = r.size
        if m < 2:
            continue
        cnt = np.zeros(V)
        for x in r: cnt[v_index[x]] += 1
        for i in range(V):
            for j in range(V):
                o[i, j] += (cnt[i]*(cnt[i]-1) if i == j else cnt[i]*cnt[j]) / (m-1)
    n_v = o.sum(1); n = n_v.sum()
    if n < 2:
        return np.nan
    def d2(a, b):
        if level == "nominal": return 0.0 if a == b else 1.0
        if level == "interval": return (values[a]-values[b])**2
        lo, hi = (a, b) if a <= b else (b, a)
        return (n_v[lo:hi+1].sum() - (n_v[lo]+n_v[hi])/2)**2
    D_o = sum(o[i, j]*d2(i, j) for i in range(V) for j in range(V)) / n
    D_e = sum(n_v[i]*n_v[j]*d2(i, j) for i in range(V) for j in range(V)) / (n*(n-1))
    return np.nan if D_e == 0 else 1 - D_o/D_e


def cohen_kappa(a, b, labels=None, weights=None):
    a = np.asarray(a); b = np.asarray(b)
    if labels is None: labels = np.unique(np.concatenate([a, b]))
    K = len(labels); idx = {v: i for i, v in enumerate(labels)}
    O = np.zeros((K, K))
    for x, y in zip(a, b): O[idx[x], idx[y]] += 1
    N = O.sum()
    if N == 0: return np.nan
    E = np.outer(O.sum(1), O.sum(0)) / N
    if weights is None:
        w = 1 - np.eye(K)
    else:
        i, j = np.indices((K, K)); d = np.abs(i-j).astype(float)
        w = d if weights == "linear" else d**2
        w = w/w.max() if w.max() > 0 else w
    exp = (w*E).sum()
    return np.nan if exp == 0 else 1 - (w*O).sum()/exp


def cronbach_alpha(matrix):
    m = np.asarray(matrix, dtype=float); m = m[~np.isnan(m).any(axis=1)]
    k = m.shape[1]
    if k < 2 or m.shape[0] < 2: return np.nan
    tv = m.sum(1).var(ddof=1)
    return np.nan if tv == 0 else (k/(k-1))*(1 - m.var(0, ddof=1).sum()/tv)

### Loading, completeness, pairing

In [45]:
def _match_col(columns, target):
    for c in columns:
        if any(str(c).strip().lower().startswith(a) for a in COL_ALIASES[target]):
            return c
    raise KeyError(f"{target!r} not found in {list(columns)[:6]}")


def discover_texts(root):
    root = Path(root); texts = {}
    if root.exists():
        for d in sorted(p for p in root.iterdir() if p.is_dir()):
            if glob.glob(str(d / "*.xlsx")): texts[d.name] = d
    if not texts and glob.glob(str(root / "*.xlsx")): texts[root.name] = root
    return texts


def load_text(folder, target, exclude_groups=frozenset()):
    frames = []
    for path in sorted(glob.glob(str(Path(folder) / "*.xlsx"))):
        xls = pd.ExcelFile(path)
        for sheet in xls.sheet_names:
            m = SHEET_RE.match(sheet.strip())
            if not m: continue
            theory, group = int(m.group(1)), m.group(2).lower()
            if group in exclude_groups: continue
            df = xls.parse(sheet)
            sub = df[[df.columns[0], _match_col(df.columns, target)]].copy()
            sub.columns = ["sentence_id", "rating"]
            sub["sentence_id"] = pd.to_numeric(sub["sentence_id"], errors="coerce")
            sub["rating"] = pd.to_numeric(sub["rating"], errors="coerce")
            sub = sub.dropna(subset=["sentence_id"])
            sub["sentence_id"] = sub["sentence_id"].astype(int)
            sub = sub.assign(theory=theory, group=group, unit=f"T{theory}{group}")
            frames.append(sub)
    cols = ["sentence_id", "rating", "theory", "group", "unit"]
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=cols)


def load_all(texts, target, exclude_groups=frozenset()):
    frames = []
    for name, folder in texts.items():
        t = load_text(folder, target, exclude_groups)
        if len(t): frames.append(t.assign(text=name))
    cols = ["sentence_id", "rating", "theory", "group", "unit", "text"]
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=cols)


def unit_meta(long):
    return (long[["unit", "theory", "group"]].drop_duplicates()
            .sort_values(["theory", "group"]).reset_index(drop=True))


def rating_matrix(long, units=None):
    w = long.pivot_table(index="sentence_id", columns="unit", values="rating").sort_index()
    return w[[u for u in units if u in w.columns]] if units is not None else w


def completeness(long, min_fraction):
    recs = []
    for unit, g in long.groupby("unit"):
        n_sent = g["sentence_id"].nunique(); n_rated = int(g["rating"].notna().sum())
        var = g["rating"].dropna().var()
        recs.append(dict(unit=unit, theory=g.theory.iloc[0], group=g.group.iloc[0],
                         n_sentences=n_sent, n_rated=n_rated,
                         frac_rated=round(n_rated/max(n_sent, 1), 3),
                         constant=bool(pd.isna(var) or var == 0)))
    rep = pd.DataFrame(recs).sort_values(["theory", "group"]).reset_index(drop=True)
    rep["completed"] = rep.frac_rated >= min_fraction
    rep["included"] = rep.completed & ~rep.constant
    return rep

In [46]:
def _pair_metric(matrix, u1, u2, metric, labels, min_shared):
    a, b = matrix[u1], matrix[u2]
    if metric == "Krippendorff (ordinal)":
        sub = np.vstack([a.to_numpy(), b.to_numpy()])
        if np.sum(~np.isnan(sub).any(axis=0)) < min_shared: return np.nan
        return krippendorff_alpha(sub, "ordinal")
    common = a.dropna().index.intersection(b.dropna().index)
    if len(common) < min_shared: return np.nan
    if metric == "Spearman (rank corr.)": return spearmanr(a.loc[common], b.loc[common])[0]
    x, y = a.loc[common].astype(int), b.loc[common].astype(int)
    if metric == "Percent agreement (raw)": return float((x == y).mean())
    return cohen_kappa(x, y, labels=labels, weights=None if metric == "Cohen kappa" else "quadratic")


def pairwise_iaa(long, units, metric, min_shared):
    matrix = rating_matrix(long, units); meta = unit_meta(long).set_index("unit")
    vals = long["rating"].dropna().astype(int)
    labels = list(range(int(vals.min()), int(vals.max())+1)) if len(vals) else [0]
    sq = pd.DataFrame(np.nan, index=units, columns=units, dtype=float); rows = []
    for u1, u2 in combinations(units, 2):
        v = _pair_metric(matrix, u1, u2, metric, labels, min_shared)
        sq.loc[u1, u2] = sq.loc[u2, u1] = v
        rows.append(dict(pair=f"{u1}-{u2}",
                         label="intra" if meta.loc[u1, "theory"] == meta.loc[u2, "theory"] else "inter",
                         theory1=meta.loc[u1, "theory"], theory2=meta.loc[u2, "theory"],
                         group1=meta.loc[u1, "group"], group2=meta.loc[u2, "group"], value=v))
    for u in units: sq.loc[u, u] = 1.0
    return sq, pd.DataFrame(rows)


def overall_krippendorff(long, units, level="ordinal"):
    return krippendorff_alpha(rating_matrix(long, units).to_numpy().T, level)


def category_matrix(pair_long, key):
    k1, k2 = f"{key}1", f"{key}2"
    cats = sorted(set(pair_long[k1]) | set(pair_long[k2]))
    M = pd.DataFrame(np.nan, index=cats, columns=cats, dtype=float)
    for ci in cats:
        for cj in cats:
            mask = (((pair_long[k1] == ci) & (pair_long[k2] == cj)) |
                    ((pair_long[k1] == cj) & (pair_long[k2] == ci)))
            v = pair_long.loc[mask, "value"].dropna()
            if len(v): M.loc[ci, cj] = v.mean()
    pref = key[0].upper()
    M.index = [f"{pref}{c}" for c in cats]; M.columns = [f"{pref}{c}" for c in cats]
    return M


def analyze_iaa(long, groups, theories, metric, min_fraction, min_shared):
    rep = completeness(long, min_fraction)
    in_filter = rep.group.isin(groups) & rep.theory.isin(theories)
    completed = rep[rep.completed & in_filter]
    chance = metric in CHANCE_CORRECTED or metric == "Spearman (rank corr.)"
    keep = completed[~completed.constant] if chance else completed
    res = dict(rep=rep, n=len(keep),
               excluded_constant=(completed[completed.constant].unit.tolist() if chance else []),
               excluded_incomplete=rep[in_filter & ~rep.completed].unit.tolist())
    units = keep.unit.tolist()
    if len(units) < 2: return res
    sq, pl = pairwise_iaa(long, units, metric, min_shared)
    cu = completed.unit.tolist()
    res.update(sq=sq, pair_long=pl, theory_mat=category_matrix(pl, "theory"),
               group_mat=category_matrix(pl, "group"),
               mean_intra=pl.loc[pl.label == "intra", "value"].mean(),
               mean_inter=pl.loc[pl.label == "inter", "value"].mean(),
               overall_k=overall_krippendorff(long, cu),
               cronbach=cronbach_alpha(rating_matrix(long, cu).to_numpy()))
    return res


def plot_iaa(res, title, vmin=-0.2, vmax=1.0, cmap="RdYlGn"):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5.4),
                             gridspec_kw={"width_ratios": [1.5, 1, 1]})
    def hm(ax, M, sub):
        sns.heatmap(M, ax=ax, annot=True, fmt=".2f", cmap=cmap, vmin=vmin, vmax=vmax,
                    linewidths=.5, linecolor="white", square=True, cbar_kws={"shrink": .7},
                    mask=M.isna(), annot_kws={"size": 8})
        ax.set_title(sub, fontsize=11); ax.tick_params(labelsize=8)
    hm(axes[0], res["sq"], "unit x unit (pairwise)")
    hm(axes[1], res["theory_mat"], "theory x theory (mean)")
    hm(axes[2], res["group_mat"], "group x group (mean)")
    fig.suptitle(title, fontsize=13, y=1.02); fig.tight_layout()
    return fig

In [47]:
def coherence_pairs(master, min_fraction, min_shared, exclude_same_group_inter):
    """Spearman per (text, annotator-pair), labelled intra/inter by theory."""
    rows = []
    for text, long in master.groupby("text"):
        rep = completeness(long, min_fraction)
        units = rep[rep.included].unit.tolist()               # completed & non-constant
        meta = unit_meta(long).set_index("unit"); wide = rating_matrix(long, units)
        for u1, u2 in combinations(sorted(units), 2):
            same_theory = meta.loc[u1, "theory"] == meta.loc[u2, "theory"]
            same_group = meta.loc[u1, "group"] == meta.loc[u2, "group"]
            label = "intra" if same_theory else "inter"
            if exclude_same_group_inter and label == "inter" and same_group: continue
            a, b = wide[u1].dropna(), wide[u2].dropna()
            common = a.index.intersection(b.index)
            rho = spearmanr(a.loc[common], b.loc[common])[0] if len(common) >= min_shared else np.nan
            rows.append(dict(text=text, pair=f"{u1}-{u2}", label=label,
                             same_group=same_group, n_shared=len(common), rho=rho))
    return pd.DataFrame(rows)


def aggregate(res, statistic):
    valid = res.dropna(subset=["rho"])
    mi = valid.loc[valid.label == "intra", "rho"].mean()
    me = valid.loc[valid.label == "inter", "rho"].mean()
    diff = mi - me if statistic == "intra_minus_inter" else me - mi
    return valid, mi, me, diff


def permutation_test(valid, n_perm, seed, statistic):
    rng = np.random.default_rng(seed)
    rho = valid["rho"].to_numpy(); is_intra = (valid["label"] == "intra").to_numpy()
    n_intra = int(is_intra.sum()); n = len(rho)
    def stat(mask):
        gap = rho[mask].mean() - rho[~mask].mean()
        return gap if statistic == "intra_minus_inter" else -gap
    observed = stat(is_intra); null = np.empty(n_perm)
    for i in range(n_perm):
        idx = rng.permutation(n); mask = np.zeros(n, bool); mask[idx[:n_intra]] = True
        null[i] = stat(mask)
    count = int(np.sum(null >= observed)); p = (count + 1) / (n_perm + 1)
    return observed, p, count, null

## 1 · Who annotated what

Loads every text and reports the **n** entering each analysis: units that completed the text and
(for the reliability metrics) were not constant. This is the audit trail behind every number
below.

In [48]:
TEXTS = discover_texts(ROOT)
MASTER = load_all(TEXTS, DEFAULT_TARGET, EXCLUDE_GROUPS)

print(f"Root: {ROOT}")
print(f"Texts found: {len(TEXTS)}  ->  {list(TEXTS)}\n")

_all_groups, _all_theories = set(), set()
for text, long in MASTER.groupby("text"):
    rep = completeness(long, COMPLETENESS_MIN_FRACTION)
    inc = rep[rep.included]
    _all_groups |= set(rep.group); _all_theories |= set(rep.theory)
    by_theory = inc.groupby("theory")["unit"].apply(lambda s: ",".join(sorted(s)))
    line = "   ".join(f"T{t}[{v}]" for t, v in by_theory.items())
    print(f"{text:26} n={len(inc):2}   {line}")
    consts = rep[rep.completed & rep.constant].unit.tolist()
    incompl = rep[~rep.completed].unit.tolist()
    if consts:  print(f"{'':26} constant (excluded from chance-corrected IAA): {consts}")
    if incompl: print(f"{'':26} incomplete (excluded): {incompl}")

ALL_GROUPS = sorted(_all_groups); ALL_THEORIES = sorted(_all_theories)
print(f"\nGroups: {ALL_GROUPS}   Theories: {ALL_THEORIES}")
print(f"Target column in memory: {DEFAULT_TARGET}")

## 2 · Q1 — Reliability (inter-annotator agreement)

*How consistently do annotators apply an operationalization?* Pick a text, target column, and
metric; the three heatmaps show unit-vs-unit agreement, the theory-vs-theory means (diagonal =
agreement **within** a theory), and group-vs-group means.

**Reading the metrics.** Krippendorff's alpha (ordinal) is the headline reliability figure; it
handles ≥2 raters and missing data and respects the 0–5 scale. Weighted Cohen's kappa is the
pairwise ordinal companion. **Percent agreement (raw)** is not chance-corrected but is the only
measure defined when a unit used one value throughout — use it for the `character_anxiety_level`
column, where several units are all-zero and alpha/kappa are undefined (reported, not hidden).

> Practical guidance for tomorrow: quote Krippendorff's alpha for `reader_suspense_level`; for
> `character_anxiety_level` lean on raw agreement and note that chance-corrected reliability is
> undefined on the near-constant units.

In [49]:
import ipywidgets as W
from IPython.display import display, clear_output

w_text   = W.Dropdown(options=list(TEXTS), description="Text:")
w_target = W.Dropdown(options=["reader_suspense_level", "character_anxiety_level"],
                      value=DEFAULT_TARGET, description="Column:")
w_metric = W.Dropdown(options=["Krippendorff (ordinal)", "Cohen kappa",
                               "Cohen kappa (weighted)", "Percent agreement (raw)",
                               "Spearman (rank corr.)"],
                      description="Metric:")
w_groups = W.SelectMultiple(options=ALL_GROUPS, value=tuple(ALL_GROUPS),
                            description="Groups:", rows=min(6, len(ALL_GROUPS)))
w_theory = W.SelectMultiple(options=ALL_THEORIES, value=tuple(ALL_THEORIES),
                            description="Theories:", rows=min(6, len(ALL_THEORIES)))
w_run = W.Button(description="Run", button_style="primary", icon="play")
_out = W.Output()

def _explore(_=None):
    with _out:
        clear_output(wait=True)
        long = load_text(TEXTS[w_text.value], w_target.value, EXCLUDE_GROUPS)
        res = analyze_iaa(long, set(w_groups.value), set(w_theory.value),
                          w_metric.value, COMPLETENESS_MIN_FRACTION, MIN_SHARED_SENTENCES)
        print(f"{w_text.value}  |  {w_metric.value}  |  {w_target.value}")
        print(f"n = {res['n']} units")
        if res["excluded_constant"]:
            print(f"  constant units (undefined here -> use raw %): {res['excluded_constant']}")
        if res.get("excluded_incomplete"):
            print(f"  incomplete: {res['excluded_incomplete']}")
        if res["n"] < 2:
            print("Need at least two units. Widen the filter."); return
        print(f"mean intra = {res['mean_intra']:+.3f}   mean inter = {res['mean_inter']:+.3f}")
        print(f"overall Krippendorff (all completed) = {res['overall_k']:.3f}   "
              f"|  Cronbach = {res['cronbach']:.3f}")
        vmin = 0.0 if w_metric.value.startswith("Percent") else -0.2
        plot_iaa(res, f"{w_text.value} — {w_metric.value}", vmin=vmin); plt.show()

w_run.on_click(_explore)
display(W.HBox([W.VBox([w_text, w_target, w_metric]), w_groups, w_theory,
                W.VBox([w_run])]), _out)
_explore()

Output()

### Reliability at a glance — overall Krippendorff per text

One number per text per column: the multi-rater ordinal alpha across all completed units. This
is the compact table to bring to the meeting.

In [50]:
rows = []
for target in ["reader_suspense_level", "character_anxiety_level"]:
    for text, folder in TEXTS.items():
        long = load_text(folder, target, EXCLUDE_GROUPS)
        rep = completeness(long, COMPLETENESS_MIN_FRACTION)
        cu = rep[rep.completed].unit.tolist()
        rows.append(dict(text=text, target=target, n_units=len(cu),
                         overall_krippendorff=round(overall_krippendorff(long, cu), 3)))
iaa_summary = pd.DataFrame(rows).pivot(index="text", columns="target",
                                       values="overall_krippendorff")
print("Overall Krippendorff's alpha (ordinal), all completed units:")
display(iaa_summary)

target,character_anxiety_level,reader_suspense_level
text,,
Anderson_Hippogriff,0.112,0.085
Doyle_How_It_Happened,0.090,0.192
Gaskell_Old_Nurses_Story,0.188,0.204
Hemingway_The_Killers,0.071,0.149
London_The_Heathen,0.076,0.248
Marks_Wedding_Day,0.085,0.085
Munro_Defensive_Diamond,0.186,0.082
Nesbit_The_Semi-detached,0.103,0.127
Wharton_Miss_Mary,0.002,-0.046


## 3 · Q2 — Does theory matter? (intra- vs. inter-theory coherence)

This is the email's procedure, pooled across **all** texts (the design aggregates over texts and
pairs, then tests once):

1. For each text, form every annotator pair; label it **intra** (same theory) or **inter**.
2. Compute **Spearman rank correlation** on the sentences the pair share — do they order
   sentences the same way for suspense/anxiety?
3. Average all intra correlations and all inter correlations; the statistic is `intra − inter`.
4. A **permutation test** (N = 10,000, labels shuffled) gives a one-tailed p-value.

Spearman is used because it is invariant to how each annotator uses the 0–5 range (a constant
offset doesn't count as disagreement), and, per the email, no chance correction is applied — the
baseline here is the inter-theory pairs, not chance.

**Decisions baked in (flip in config, discuss with supervisor):** the statistic is `intra − inter`
so positive = intra more coherent (the email's sanity check; the literal wording "subtract intra
from inter" would flip the sign); `EXCLUDE_SAME_GROUP_INTER` is off, so inter pairs that happen to
share a group are kept; `EXCLUDE_GROUPS` is empty, so `T4c` participates.

Use the controls to switch the target column or toggle the same-group option, then **Run**.

In [51]:
w_c_target = W.Dropdown(options=["reader_suspense_level", "character_anxiety_level"],
                        value=DEFAULT_TARGET, description="Column:")
w_c_samegrp = W.Checkbox(value=EXCLUDE_SAME_GROUP_INTER,
                         description="Exclude same-group inter pairs")
w_c_run = W.Button(description="Run coherence test", button_style="success", icon="play")
_c_out = W.Output()

COHERENCE = {}   # filled on run, read by the answers + export cells

def _coherence(_=None):
    with _c_out:
        clear_output(wait=True)
        master = load_all(TEXTS, w_c_target.value, EXCLUDE_GROUPS)
        res = coherence_pairs(master, COMPLETENESS_MIN_FRACTION, MIN_SHARED_SENTENCES,
                              w_c_samegrp.value)
        valid, mi, me, observed = aggregate(res, STATISTIC)
        obs, p, count, null = permutation_test(valid, N_PERMUTATIONS, RANDOM_SEED, STATISTIC)

        # per-text breakdown
        bt = (valid.groupby("text")
              .apply(lambda g: pd.Series({
                  "n_intra": int((g.label == "intra").sum()),
                  "n_inter": int((g.label == "inter").sum()),
                  "mean_intra": g.loc[g.label == "intra", "rho"].mean(),
                  "mean_inter": g.loc[g.label == "inter", "rho"].mean()}))
              .reset_index())
        bt["intra_minus_inter"] = bt.mean_intra - bt.mean_inter

        COHERENCE.update(target=w_c_target.value, res=res, valid=valid,
                         mean_intra=mi, mean_inter=me, observed=observed,
                         p=p, count=count, null=null, by_text=bt,
                         exclude_same_group=w_c_samegrp.value)

        print(f"Target: {w_c_target.value}   |   texts pooled: {valid.text.nunique()}   "
              f"|   pairs used: {len(valid)} "
              f"(intra {int((valid.label=='intra').sum())}, inter {int((valid.label=='inter').sum())})")
        print(f"\nmean intra Spearman = {mi:+.4f}")
        print(f"mean inter Spearman = {me:+.4f}")
        print(f"observed {STATISTIC} = {observed:+.4f}")
        print(f"permutation test: {count}/{N_PERMUTATIONS} >= observed  ->  one-tailed p = {p:.4f}")
        print("\nPer-text breakdown:")
        display(bt.round(3))

        fig, ax = plt.subplots(1, 2, figsize=(14, 4.2))
        ax[0].hist(null, bins=50, color="#9db4c0", edgecolor="white")
        ax[0].axvline(observed, color="#c1666b", lw=2, label=f"observed={observed:.3f}\np={p:.3f}")
        ax[0].set_title("Permutation null vs. observed"); ax[0].set_xlabel(STATISTIC)
        ax[0].set_ylabel("count"); ax[0].legend()
        order = bt.sort_values("intra_minus_inter")
        colors = ["#4a7c59" if v >= 0 else "#c1666b" for v in order.intra_minus_inter]
        ax[1].barh(order.text, order.intra_minus_inter, color=colors)
        ax[1].axvline(0, color="black", lw=.8); ax[1].set_title("intra - inter per text")
        ax[1].set_xlabel("mean Spearman difference")
        plt.tight_layout(); plt.show()

w_c_run.on_click(_coherence)
display(W.HBox([w_c_target, w_c_samegrp, w_c_run]), _c_out)
_coherence()

Output()

## 4 · Written Summary with Filled-in Numbers

Run this after the coherence cell above; it turns the numbers into plain sentences.

In [52]:
def verdict():
    if not COHERENCE:
        print("Run the coherence cell (section 3) first."); return
    C = COHERENCE
    direction = ("more similarly" if C["mean_intra"] > C["mean_inter"] else "no more similarly")
    sig = C["p"] < 0.05
    print("="*72)
    print("Q2 — Does sharing a theory make annotators agree more?")
    print("="*72)
    print(f"Target: {C['target']}   (pooled over {C['valid'].text.nunique()} texts, "
          f"{len(C['valid'])} annotator-pair observations)")
    print()
    print(f"Same-theory annotator pairs ranked sentences {direction} than different-theory")
    print(f"pairs: mean Spearman rho was {C['mean_intra']:+.3f} within a theory versus "
          f"{C['mean_inter']:+.3f} across")
    print(f"theories, a difference of {C['observed']:+.3f}.")
    print()
    print(f"A permutation test (N={N_PERMUTATIONS:,}, labels shuffled) placed the observed")
    print(f"difference at one-tailed p = {C['p']:.4f}.")
    print()
    if C["mean_intra"] > C["mean_inter"] and sig:
        print("=> The intra > inter pattern holds AND is statistically significant (p < .05):")
        print("   the four operationalizations produce distinguishable annotation behaviour.")
    elif C["mean_intra"] > C["mean_inter"] and not sig:
        print("=> The direction is as hoped (intra > inter, sanity check passes) but it is NOT")
        print(f"   significant at .05 (p = {C['p']:.3f}). Consistent with a real-but-small effect")
        print("   that needs more data/texts, or with theories overlapping in what they capture.")
    else:
        print("=> Same-theory pairs did NOT agree more than different-theory pairs. The design")
        print("   assumption is not supported for this column; worth discussing why.")
    print()
    n_neg = int((COHERENCE['by_text'].intra_minus_inter < 0).sum())
    print(f"Consistency across texts: {len(COHERENCE['by_text']) - n_neg} of "
          f"{len(COHERENCE['by_text'])} texts show intra > inter.")
    print()
    print("Caveats to raise:")
    print(" - Sign convention is intra - inter (email's sanity-check reading).")
    print(f" - Same-group inter pairs {'excluded' if C['exclude_same_group'] else 'included'}; "
          "toggling this is a robustness check.")
    print(" - On character_anxiety_level, constant units drop out and reliability is unstable;")
    print("   read Q2 there with care and pair it with the raw-agreement view from section 2.")

verdict()

## 5 · Export everything

Writes a timestamped folder with the reliability heatmaps (every text × metric), the coherence
results and figures, one results workbook, and the full run log.

In [53]:
def export_all(target=DEFAULT_TARGET):
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    out = SAVE_DIR / f"run_{stamp}"; (out / "figures").mkdir(parents=True, exist_ok=True)
    TAGS = {"Krippendorff (ordinal)": "krippendorff", "Cohen kappa": "cohen",
            "Cohen kappa (weighted)": "cohen_w", "Percent agreement (raw)": "pct_agree"}
    comp_frames, iaa_rows = [], []

    with pd.ExcelWriter(out / "results.xlsx", engine="openpyxl") as xw:
        # --- Q1 reliability: heatmaps + matrices per text/metric
        for text, folder in TEXTS.items():
            long = load_text(folder, target, EXCLUDE_GROUPS)
            rep = completeness(long, COMPLETENESS_MIN_FRACTION); rep.insert(0, "text", text)
            comp_frames.append(rep)
            for metric in ["Krippendorff (ordinal)", "Cohen kappa (weighted)",
                           "Percent agreement (raw)"]:
                res = analyze_iaa(long, set(ALL_GROUPS), set(ALL_THEORIES), metric,
                                  COMPLETENESS_MIN_FRACTION, MIN_SHARED_SENTENCES)
                if res["n"] < 2: continue
                tag = TAGS[metric]; st = text[:18]
                res["theory_mat"].to_excel(xw, sheet_name=f"{st}_{tag}_theory"[:31])
                vmin = 0.0 if metric.startswith("Percent") else -0.2
                fig = plot_iaa(res, f"{text} — {metric}", vmin=vmin)
                fig.savefig(out / "figures" / f"IAA_{text}__{tag}.png", dpi=150, bbox_inches="tight")
                plt.close(fig)
                iaa_rows.append(dict(text=text, metric=metric, n=res["n"],
                                     mean_intra=res["mean_intra"], mean_inter=res["mean_inter"],
                                     overall_krippendorff=res["overall_k"], cronbach=res["cronbach"],
                                     excluded_constant=",".join(res["excluded_constant"])))
        pd.concat(comp_frames, ignore_index=True).to_excel(xw, sheet_name="completeness", index=False)
        pd.DataFrame(iaa_rows).to_excel(xw, sheet_name="iaa_summary", index=False)

        # --- Q2 coherence (uses last COHERENCE run, or computes on `target`)
        if COHERENCE:
            C = COHERENCE
        else:
            master = load_all(TEXTS, target, EXCLUDE_GROUPS)
            r = coherence_pairs(master, COMPLETENESS_MIN_FRACTION, MIN_SHARED_SENTENCES,
                                EXCLUDE_SAME_GROUP_INTER)
            v, mi, me, ob = aggregate(r, STATISTIC)
            ob, p, cnt, nul = permutation_test(v, N_PERMUTATIONS, RANDOM_SEED, STATISTIC)
            C = dict(target=target, res=r, valid=v, mean_intra=mi, mean_inter=me,
                     observed=ob, p=p, null=nul,
                     by_text=v.groupby("text").apply(lambda g: pd.Series({
                         "mean_intra": g.loc[g.label=="intra","rho"].mean(),
                         "mean_inter": g.loc[g.label=="inter","rho"].mean()})).reset_index())
        C["res"].to_excel(xw, sheet_name="coherence_pairs", index=False)
        C["by_text"].round(4).to_excel(xw, sheet_name="coherence_by_text", index=False)
        pd.DataFrame([dict(target=C["target"], statistic=STATISTIC,
                           mean_intra=C["mean_intra"], mean_inter=C["mean_inter"],
                           observed=C["observed"], p_value=C["p"],
                           n_permutations=N_PERMUTATIONS)]).to_excel(
            xw, sheet_name="coherence_overall", index=False)

        fig, ax = plt.subplots(figsize=(7, 4))
        ax.hist(C["null"], bins=50, color="#9db4c0", edgecolor="white")
        ax.axvline(C["observed"], color="#c1666b", lw=2,
                   label=f"observed={C['observed']:.3f}\np={C['p']:.3f}")
        ax.set_title(f"Coherence permutation test — {C['target']}")
        ax.set_xlabel(STATISTIC); ax.legend(); fig.tight_layout()
        fig.savefig(out / "figures" / "coherence_permutation.png", dpi=150, bbox_inches="tight")
        plt.close(fig)

    (out / "run_log.txt").write_text(_output_log.getvalue(), encoding="utf-8")
    print(f"Saved to: {out.resolve()}")
    print(f"  figures/  ({len(list((out/'figures').glob('*.png')))} PNGs)")
    print("  results.xlsx  (completeness, iaa_summary, per-text matrices, coherence_*)")
    print("  run_log.txt")
    return out

saved = export_all()

## Summary (one-paragraph version)

- **Reliability (Q1):** report the overall Krippendorff table from section 2 for
  `reader_suspense_level`; note that `character_anxiety_level` is too zero-inflated for
  chance-corrected reliability and is better described with raw agreement.
- **Does theory matter (Q2):** the section 4 readout gives the exact sentence — whether
  same-theory annotators ranked sentences more similarly than different-theory annotators, by how
  much, and whether the permutation test makes it significant, plus how many texts share the
  direction.
- **Robustness:** flip `EXCLUDE_SAME_GROUP_INTER` and re-run section 3 to show the effect isn't
  driven by same-annotator/different-theory pairs; switch the column to show it holds (or not) for
  anxiety.

If helpful next, I can add Gwet's AC2 (robust to the anxiety column's skew) and a leave-one-out
alpha to flag individual annotators — the two pieces this notebook doesn't yet compute.